# Campagne 1 : méthodes d'ensemble sur les facteurs booléens

Clone le dépôt public au commit `COMMIT_ATTENDU`, installe les versions exactes de `requirements.txt` (calcul seul, sans les outils de test), décrit la machine, reprend éventuellement une session précédente, lance la campagne (arrêt propre à `DUREE_MAX`), assemble et archive les résultats.

Lancer par **Save Version → Save & Run All**, Internet activé. Aucun secret : le dépôt est public.
Les résultats sont dans `/kaggle/working/depot/resultats/<dossier>/` et dans l'archive `/kaggle/working/<dossier>.zip`.

In [ ]:
CONFIG = "experiences/essai_kaggle.yaml"   # puis "experiences/campagne1.yaml"
COMMIT_ATTENDU = ""                        # identifiant du commit à exécuter (au moins 7 caractères)
REPRISE = None                             # ou le dossier de résultats d'une session précédente, sous /kaggle/input/
DUREE_MAX = "11h"
DEPOT_URL = "https://github.com/mentalist011101/campagne1_extraction-de-facteur.git"

import os
import shutil
import subprocess
import sys
from pathlib import Path

DEPOT = Path("/kaggle/working/depot")
os.environ["PYTHONUNBUFFERED"] = "1"


def executer(commande, verifier=True):
	print("$", " ".join(map(str, commande)), flush=True)
	processus = subprocess.Popen(commande, cwd=DEPOT if DEPOT.exists() else None, stdout=subprocess.PIPE,
		stderr=subprocess.STDOUT, text=True)
	for ligne in processus.stdout:
		print(ligne, end="", flush=True)
	code = processus.wait()
	if verifier and code != 0:
		raise RuntimeError(f"Échec (code {code}) : {' '.join(map(str, commande))}")
	return code

In [ ]:
if len(COMMIT_ATTENDU) < 7:
	raise ValueError("Renseigner COMMIT_ATTENDU (identifiant du commit, au moins 7 caractères).")
if DEPOT.exists():
	raise RuntimeError(f"{DEPOT} existe déjà : relancer dans une session neuve.")
executer(["git", "clone", "--quiet", DEPOT_URL, str(DEPOT)])
executer(["git", "checkout", "--quiet", COMMIT_ATTENDU])
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=DEPOT, capture_output=True, text=True).stdout.strip()
if not commit.startswith(COMMIT_ATTENDU):
	raise RuntimeError(f"Commit cloné {commit}, attendu {COMMIT_ATTENDU}.")
executer(["git", "log", "-1", "--format=Commit cloné : %H%n%s%n%ad"])
etat = subprocess.run(["git", "status", "--porcelain"], cwd=DEPOT, capture_output=True, text=True).stdout.strip()
print("Dépôt propre." if not etat else f"ATTENTION : dépôt modifié :\n{etat}")

In [ ]:
executer([sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements.txt"])

In [ ]:
# Versions réellement installées et description de la machine, dans <dossier>/sessions/<horodatage>.txt.
# S'arrête si une version diffère de requirements.txt.
executer([sys.executable, "-m", "experiences.session_kaggle", "decrire", "--config", CONFIG])

In [ ]:
if REPRISE is None:
	print("Pas de reprise : session neuve.")
elif not Path(REPRISE).is_dir():
	for chemin in sorted(Path("/kaggle/input").glob("*/*/*/*")):
		print(chemin)
	raise FileNotFoundError(f"REPRISE introuvable : {REPRISE} (dossiers disponibles ci-dessus)")
else:
	executer([sys.executable, "-m", "experiences.session_kaggle", "reprendre", "--config", CONFIG, "--source", REPRISE])

In [ ]:
# timeout coupe la campagne à DUREE_MAX, avant la limite de Kaggle : code 124, arrêt normal, la suite
# (assemblage, archive) s'exécute et la sortie de la version est gardée. Tout autre code non nul : erreur.
# L'écriture des plis est atomique : un arrêt ne perd que le pli en cours.
executer([sys.executable, "-m", "experiences.session_kaggle", "lancer", "--config", CONFIG, "--duree-max", DUREE_MAX])

In [ ]:
import csv

import yaml

dossier = DEPOT / yaml.safe_load((DEPOT / CONFIG).read_text(encoding="utf-8"))["dossier_resultats"]
executer([sys.executable, "-m", "experiences.campagne1", "--config", CONFIG, "--assembler"])
vus = set()
with open(dossier / "campagne1_par_jeu.csv", encoding="utf-8") as fichier:
	for ligne in csv.DictReader(fichier):
		if ligne["jeu"] not in vus:
			vus.add(ligne["jeu"])
			plis = len(list((dossier / ligne["jeu"]).glob("pli_*.csv")))
			print(f"{ligne['jeu']:16s} {ligne['statut']:18s} {plis} plis")

In [ ]:
archive = shutil.make_archive(f"/kaggle/working/{dossier.name}", "zip", root_dir=dossier.parent, base_dir=dossier.name)
print(archive, f"{Path(archive).stat().st_size / 1e6:.2f} Mo")